In [1]:
import sqlite3
import pandas as pd

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

In [2]:
conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    status TEXT,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_lines (
    line_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    FOREIGN KEY(order_id) REFERENCES orders(order_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER UNIQUE,
    payment_date TEXT,
    method TEXT,
    amount REAL,
    status TEXT,
    FOREIGN KEY(order_id) REFERENCES orders(order_id)
);
""")


In [4]:
# Customers
customers = [
    (1, 'Alice', 'Smith'),
    (2, 'Bob', 'Johnson'),
    (3, 'Carol', 'White')
]

conn.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)

# Orders
orders = [
    (1, 1, '2024-01-01', 'Delivered'),
    (2, 1, '2024-01-02', 'Delivered'),
    (3, 2, '2024-01-03', 'Delivered'),
    (4, 3, '2024-01-04', 'Delivered')
]

conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)

# Order Lines
order_lines = [
    (1, 1, 101, 2, 100.00),
    (2, 1, 102, 1, 50.00),
    (3, 2, 103, 3, 200.00),
    (4, 3, 104, 2, 150.00),
    (5, 4, 105, 1, 300.00)
]

conn.executemany("INSERT INTO order_lines VALUES (?, ?, ?, ?, ?)", order_lines)


In [6]:
query = """
SELECT
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(ol.quantity * ol.unit_price), 2) AS total_spent,
    RANK() OVER (ORDER BY SUM(ol.quantity * ol.unit_price) DESC) AS customer_rank
FROM customers c
JOIN orders o 
    ON c.customer_id = o.customer_id
JOIN order_lines ol 
    ON o.order_id = ol.order_id
GROUP BY c.customer_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY customer_rank;
"""

df_result = pd.read_sql_query(query, conn)
df_result


,customer_name,total_orders,total_spent,customer_rank
0,Alice Smith,2,850.0,1
1,Carol White,1,300.0,2
2,Bob Johnson,1,300.0,2


In [7]:
pd.read_sql_query("SELECT * FROM customers", conn)


,customer_id,first_name,last_name
0,1,Alice,Smith
1,2,Bob,Johnson
2,3,Carol,White
